# Simple Agentic RAG

This notebook walks through building an Agentic RAG system using LangChain and FAISS.

**Steps:**
1. Install dependencies
2. Ingest documents into a FAISS vector store
3. Build a LangChain agent with a document search tool
4. Ask questions

## 1. Install Dependencies

In [1]:
import os

if not os.path.exists('requirements.txt'):
    !wget https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-agentic-01/requirements.txt
else:
    print('requirements.txt already exists.')
%pip install -r  requirements.txt


--2026-06-05 06:42:02--  https://raw.githubusercontent.com/kumarsirish/FDP-AGENENTIC-AI-RAG/main/rag-agentic-01/requirements.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 196 [text/plain]
Saving to: ‘requirements.txt’

requirements.txt    100%[===================>]     196  --.-KB/s    in 0s      

2026-06-05 06:42:03 (8.10 MB/s) - ‘requirements.txt’ saved [196/196]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 33.4 MB/s eta 0:00:

## 2. Set API Keys

In [2]:
import os
from dotenv import load_dotenv


# Fallback: try Google Colab secrets
try:
    from google.colab import userdata
    if not os.getenv("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if not os.getenv("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY") or ""
except ImportError:
    # Load from .env file (local development)
    load_dotenv("/home/sirkumar/FDP-AGENENTIC-AI-RAG/.env")

HF_TOKEN = os.getenv("HF_TOKEN")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# LangChain Google GenAI requires GOOGLE_API_KEY
#if GEMINI_API_KEY:
#    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
print(f"HF_TOKEN set:      {bool(HF_TOKEN)}")
print(f"GEMINI_API_KEY set: {bool(GEMINI_API_KEY)}")

HF_TOKEN set:      True
GEMINI_API_KEY set: True


In [3]:
department_info = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",

    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science.",

]


## 3. Ingest Documents into FAISS Vector Store

Defines knowledge inline, splits it into chunks, embeds them, and saves the vector store locally.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

documents = [Document(page_content=text) for text in department_info]

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
docs = splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2")
db = FAISS.from_documents(docs, embeddings)
db.save_local("vectorstore")

print(f"Vector DB created successfully ({len(docs)} chunks)")

/tmp/ipykernel_504/2881003084.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB created successfully (8 chunks)


In [5]:
# Create retriever — returns top 2 most similar documents for any query
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 2})
print("Retriever ready.")

Retriever ready.


## 4. Build the RAG Agent

Loads the vector store and wires it up as a tool for a LangChain ZERO_SHOT_REACT agent.

In [6]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model


db = FAISS.load_local(
    "vectorstore",
    embeddings,
    allow_dangerous_deserialization=True
)

@tool
def search_answers(query: str) -> str:
    """Search the knowledge base"""
    results = retriever.invoke(query)
    print(f" Found {results}")
    docs= "\n".join([doc.page_content for doc in results])
    # Use StrOutputParser to ensure clean string output
    return StrOutputParser().invoke(docs)


# Switch providers via env vars (set before running this cell):
#   google_genai → MODEL_NAME=google_genai:gemini-2.5-flash  (default)
#   openai       → MODEL_NAME=openai:gpt-4o
#   anthropic    → MODEL_NAME=anthropic:claude-3-5-sonnet-20241022
#   ollama       → MODEL_NAME=ollama:llama3
#   huggingface  → MODEL_NAME= huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0
MODEL_NAME = "google_genai:gemini-2.5-flash-lite"
model_api_key = os.getenv("GEMINI_API_KEY")
MODEL_NAME = "huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model_api_key = os.getenv("HF_TOKEN")


chat_llm = init_chat_model(
    MODEL_NAME,
    api_key=model_api_key
)

agent = create_agent(
    model=chat_llm,
    tools=[search_answers],
    system_prompt=(
        "Your goal is to answer the user's question. "
        "First, use the 'search_answers' tool to retrieve relevant information from the knowledge base. "
        "Then, use the retrieved information to formulate a comprehensive and concise answer to the user's question. "
        "If the search results do not contain enough information to answer the question, "
        "state that you cannot answer the question based on the provided knowledge base."
    ),
)

print(f"Agent ready (model: {MODEL_NAME}).")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Agent ready (model: huggingface:TinyLlama/TinyLlama-1.1B-Chat-v1.0).


## 5. Ask Questions

Edit `question` below and re-run this cell to query the agent.

In [11]:
import pprint
def ask(question: str):
    return agent.invoke({"messages": [{"role": "user", "content": question}]})


response = ask("What is DQE?")
pprint.pprint(response)
#print(response['messages'][-1].page_content)
print("\nAgent's Answer:", response['messages'][2].content)

{'messages': [HumanMessage(content='What is DQE?', additional_kwargs={}, response_metadata={}, id='a3a2932c-675d-412d-bcee-a91cfebc554e'),
              AIMessage(content="<|system|>\nYour goal is to answer the user's question. First, use the 'search_answers' tool to retrieve relevant information from the knowledge base. Then, use the retrieved information to formulate a comprehensive and concise answer to the user's question. If the search results do not contain enough information to answer the question, state that you cannot answer the question based on the provided knowledge base.</s>\n<|user|>\nWhat is DQE?</s>\n<|assistant|>\nDQE, or Decision Quality Enhancement, is a software development approach that aims to improve the quality and efficiency of software development by enhancing the decision-making process. It involves a set of activities and practices designed to help developers identify and resolve potential issues before they result in delays and errors in the development pro

IndexError: list index out of range

In [ ]:
response = ask("How many students does DQE have?")
print("\nAgent's Answer:", response['messages'][2].content)

In [ ]:
response = ask("Whats the objective of DQE department")
print("\nAgent's Answer:", response['messages'][2].content)